In [ ]:
# ▶ Colab setup — run this cell first. (On BinderHub it does nothing.)
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/gen-hep-notebooks'):
        !git clone -b deploy --depth 1 https://github.com/livaage/gen-hep-notebooks.git /content/gen-hep-notebooks
        !pip install -q -r /content/gen-hep-notebooks/requirements-colab.txt
    sys.path.insert(0, '/content/gen-hep-notebooks')
    os.chdir('/content/gen-hep-notebooks/notebooks')

<div style="border-left: 4px solid #2563eb; background: #f8fafc; padding: 14px 18px; margin: 6px 0; border-radius: 4px; color: #1f2937;"><div style="font-weight: 600; color: #1e3a8a; margin-bottom: 8px; font-size: 1.05em;">👋 Welcome — quick reference for Cadence</div><ul style="margin: 0; padding-left: 22px; line-height: 1.65; color: #1f2937;"><li><strong>Submit an answer:</strong> each exercise cell ends with <code>check("id", answer)</code> — replace the placeholder with your answer and run the cell.</li><li><strong>Submit a plot:</strong> <code>submit_image("id", fig)</code> ships a matplotlib or Plotly figure to your teacher's dashboard.</li><li><strong>Free-text / reflections:</strong> write your response, then <code>mark_done("id")</code> to mark it complete.</li><li><strong>Stuck?</strong> <code>show_hint("id")</code> for the teacher's hint, or <code>show_solution("id")</code> after a few wrong attempts if your teacher enabled solution reveals.</li><li><strong>Your data:</strong> <code>%cadence_export_my_data</code> to download what's stored about you, <code>%cadence_delete_my_data --yes</code> to wipe it.</li></ul></div>

In [ ]:
%load_ext cadence
%cadence_session sage-linen-42 "your name"
from cadence import check, show_hint, show_solution, mark_done, submit_image

# 02 · Generative Adversarial Networks

**Generative Modelling for HEP** — notebook 2 of 6.

A GAN is a two-player game: a **generator** turns noise into fake samples, and a
**discriminator** (or, for WGAN, a **critic**) tries to tell real from fake. They
train against each other until the fakes look real. It is powerful but famously
temperamental — the headline failure is **mode collapse**, where the generator
gives up on diversity and produces the *same* output over and over.

We build intuition on **QuickDraw doodles** 🖊️: first a vanilla GAN, then we
*deliberately* drive it into mode collapse, then we *fix* it with **WGAN-GP**.
Finally we carry the idea to physics with a **point-cloud jet GAN** and watch
our recurring **jet-mass** spine — where the GAN nails the bulk but drops the
high-mass tail.

> You complete the cells marked **Exercise** — most ask for a number or a
> line of code, a few ask for a short **written reflection**. Everything else
> runs as-is.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import sys; sys.path.insert(0, "..")
from src.seeds import set_seed
from src.train import get_device, to_loader, train
from src.data import load_quickdraw_local, load_jetnet
from src.gnn import DeepSetsEncoder, DeepSetsDecoder
from src.jets import JetStandardizer
from src.jetmass import jet_mass, plot_jet_mass, jet_mass_w1, plot_jets, plot_jet_overlay

SEED = 0
NOISE_DIM = 32
# GANs need many epochs to draw recognisable doodles. 20 is a usable default on
# GPU (~1-2 min/training); on a CPU drop to ~5 (the *concepts* — adversarial
# loss, mode collapse, the WGAN-GP fix — all still come through, the samples
# just stay rough); on GPU push to 50+ for crisp doodles, or load a checkpoint.
EPOCHS = 5
BATCH = 128
set_seed(SEED)
device = get_device()
print(f"device={device}  noise_dim={NOISE_DIM}  epochs={EPOCHS}")

## Part A · A GAN on QuickDraw doodles

We load real Google **QuickDraw** bitmaps at 28×28 — four categories
(cat / fish / apple / bicycle), which double as the *modes* we'll track for mode
collapse. We use a small **DCGAN** (convolutional) generator and discriminator —
conv nets draw far sharper doodles than a plain fully-connected net.

In [ ]:
images, mode_labels = load_quickdraw_local(n=4000)         # (N,1,28,28), real doodles
X = torch.as_tensor(images, dtype=torch.float32)
img_loader = to_loader(X, batch_size=BATCH)
IMG_SHAPE = tuple(X.shape[1:])                             # (1, 28, 28)
IMG_DIM = int(np.prod(IMG_SHAPE))
N_MODES = int(len(np.unique(mode_labels)))                # categories -> "modes"
print("images:", IMG_SHAPE, "| modes (categories):", N_MODES)


class Generator(nn.Module):
    """DCGAN generator: noise -> 7x7 feature map -> upsample to 28x28."""
    def __init__(self, noise=NOISE_DIM):
        super().__init__()
        self.fc = nn.Linear(noise, 128 * 7 * 7)
        self.net = nn.Sequential(
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),  # 7 -> 14
            nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Sigmoid())                     # 14 -> 28

    def forward(self, z):
        return self.net(self.fc(z).view(-1, 128, 7, 7))


class Discriminator(nn.Module):
    """DCGAN discriminator: 28x28 -> conv downsample -> logit."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),       # 28 -> 14
            nn.Conv2d(64, 128, 4, 2, 1), nn.LeakyReLU(0.2),     # 14 -> 7
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))            # logit (BCEWithLogits)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def sample_noise(n):
    return torch.randn(n, NOISE_DIM, device=device)


# A peek at the real doodles we're trying to imitate.
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, im in zip(axes, X[np.random.permutation(len(X))[:8]]):
    ax.imshow(im.squeeze(0).numpy(), cmap="gray"); ax.axis("off")
fig.suptitle("real QuickDraw doodles"); plt.show()
print("Generator params:", sum(p.numel() for p in Generator().parameters()))

<!-- cadence:task ex1-adversarial-loss -->
### Exercise 1 — The adversarial loss

A vanilla (non-saturating) GAN alternates two updates per batch:

* **discriminator**: push real → 1, fake → 0
  $\;\mathcal{L}_D = \text{BCE}(D(x),1) + \text{BCE}(D(G(z)),0)$
* **generator**: fool the discriminator
  $\;\mathcal{L}_G = \text{BCE}(D(G(z)),1)$

Fill in the two loss terms in `gan_step`, run a few epochs of the manual
training loop (provided), and report the **final discriminator loss** as
`final_d_loss` (a number).

In [ ]:
bce = nn.BCEWithLogitsLoss()

def gan_step(G, D, real, opt_g, opt_d):
    n = real.shape[0]
    ones, zeros = torch.ones(n, device=device), torch.zeros(n, device=device)

    # --- discriminator: score reals toward 1 and fakes toward 0 ---
    fake = G(sample_noise(n)).detach()
    d_loss = ...        # add the two BCE terms: reals-vs-ones and fakes-vs-zeros
    opt_d.zero_grad(); d_loss.backward(); opt_d.step()

    # --- generator: make D score the fakes as real (fake -> 1) ---
    gen = G(sample_noise(n))
    g_loss = ...        # bce(D(gen), ones)
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()
    return float(d_loss), float(g_loss)

# Training loop (given) — runs once you've filled the two losses above.
set_seed(SEED)
G = Generator().to(device)
D = Discriminator().to(device)
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_hist = []
for _ in range(EPOCHS):
    epoch_d = [gan_step(G, D, real.to(device), opt_g, opt_d)[0] for (real,) in img_loader]
    d_hist.append(float(np.mean(epoch_d)))
final_d_loss = round(d_hist[-1], 3)
print("final discriminator loss:", final_d_loss)

check("ex1-adversarial-loss", final_d_loss)

### Does it draw doodles?

Real doodles (top) vs the vanilla GAN's samples (bottom) after training. At a few
epochs they're rough; with more epochs (bump `EPOCHS`) recognisable shapes
emerge. GANs are finicky and slow to train — that's part of today's lesson, and
why we'll fix the failure modes next.

In [ ]:
G.eval()
with torch.no_grad():
    gen = G(sample_noise(8)).cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j in range(8):
    axes[0, j].imshow(X[j, 0], cmap="gray"); axes[0, j].set_xticks([]); axes[0, j].set_yticks([])
    axes[1, j].imshow(gen[j, 0], cmap="gray"); axes[1, j].set_xticks([]); axes[1, j].set_yticks([])
axes[0, 0].set_ylabel("real"); axes[1, 0].set_ylabel("GAN")
fig.suptitle("real doodles (top) vs vanilla GAN samples (bottom)")
plt.show()

<!-- cadence:task ex2-equilibrium -->
### Exercise 2 — What does the discriminator loss tell you? (reflection)

Look at the number you just printed: the discriminator loss settles near **0.69**
— that's ln 2. Write 2–3 sentences and submit:

- A perfect classifier drives its loss toward 0. Why does a *well-matched* GAN's
  discriminator instead hover around ln 2 ≈ 0.69?
- What does that imply about using the loss curve to judge how good the samples
  are?

In [ ]:
# Your reflection (2-3 sentences), then run the cell to submit:
#   * why does a well-matched GAN's discriminator loss sit near ln 2 ≈ 0.69?
#   * what does that mean for reading GAN loss curves to judge sample quality?

check("ex2-equilibrium", "...your 2-3 sentences here...")

## Part B · Mode collapse — "the GAN only draws one thing"

Now we *induce* the classic pathology. By over-training the generator relative to
the discriminator (many G steps per D step) on a tiny noise budget, the generator
learns to dump all its mass on whichever mode currently fools D best — it stops
covering the others. We retrain a deliberately collapse-prone GAN for you below.

In [ ]:
# A collapse-prone training schedule: hammer the generator, starve the critic.
set_seed(SEED)
G_collapse = Generator().to(device)
D_collapse = Discriminator().to(device)
og = torch.optim.Adam(G_collapse.parameters(), lr=2e-3, betas=(0.5, 0.999))
od = torch.optim.Adam(D_collapse.parameters(), lr=5e-5, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
for _ in range(EPOCHS):
    for (real,) in img_loader:
        real = real.to(device)
        n = real.shape[0]
        # one stingy discriminator step
        fake = G_collapse(sample_noise(n)).detach()
        dl = bce(D_collapse(real), torch.ones(n, device=device)) + \
             bce(D_collapse(fake), torch.zeros(n, device=device))
        od.zero_grad(); dl.backward(); od.step()
        # FIVE greedy generator steps -> collapse toward one mode
        for _g in range(5):
            gen = G_collapse(sample_noise(n))
            gl = bce(D_collapse(gen), torch.ones(n, device=device))
            og.zero_grad(); gl.backward(); og.step()


def nearest_mode(samples, refs, ref_modes):
    """Assign each generated sample to the data mode of its nearest real
    neighbour (a tiny coverage proxy)."""
    s = samples.reshape(len(samples), -1)
    r = refs.reshape(len(refs), -1)
    d = torch.cdist(s, r)                       # (n_samples, n_refs)
    nn_idx = d.argmin(dim=1).cpu().numpy()
    return ref_modes[nn_idx]


print("collapse-prone GAN trained;  helper `nearest_mode` ready")

<!-- cadence:task ex3-mode-collapse -->
### Exercise 3 — Measure the mode collapse

Generate 512 doodles from the collapsed generator, snap each to the **nearest
real-data mode** (use the provided `nearest_mode`), and count how many of the
`N_MODES` data modes the generator still covers. A healthy GAN covers all of
them; a collapsed one covers just one or two. Return the integer count as
`modes_covered` (a number).

In [ ]:
G_collapse.eval()
with torch.no_grad():
    samples = G_collapse(sample_noise(512))
    refs = X[:1024].to(device)
    ref_modes = mode_labels[:1024]
    assigned = nearest_mode(samples, refs, ref_modes)   # data-mode id for each fake
    # how many of the N_MODES data modes still appear among the fakes?
    modes_covered = ...     # int(len(np.unique(assigned)))
print(f"collapsed GAN covers {modes_covered} of {N_MODES} modes")

check("ex3-mode-collapse", modes_covered)

In [ ]:
# the collapsed GAN's samples — notice how alike they look (little diversity):
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, im in zip(axes, samples[:8].cpu()):
    ax.imshow(im.squeeze(0).numpy(), cmap="gray"); ax.axis("off")
fig.suptitle(f"collapsed GAN samples — covers {modes_covered}/{N_MODES} modes")
plt.show()

## Part C · Fixing it with WGAN-GP

The Wasserstein GAN replaces the BCE classifier with a **critic** that scores
real vs fake, trained to be **1-Lipschitz** via a **gradient penalty** on samples
interpolated between real and fake:

$$\text{GP} = \lambda\,\mathbb{E}_{\hat x}\big[(\lVert\nabla_{\hat x} C(\hat x)\rVert_2 - 1)^2\big],
\qquad \hat x = \epsilon x + (1-\epsilon)\tilde x .$$

This gives smooth, informative gradients everywhere, which is exactly what kills
mode collapse. We provide a critic and a WGAN training loop; you implement the
gradient penalty.

In [ ]:
class Critic(nn.Module):
    """Like the discriminator but outputs a raw score (no sigmoid), and NO
    BatchNorm — batch stats would break the per-sample gradient penalty."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),       # 28 -> 14
            nn.Conv2d(64, 128, 4, 2, 1), nn.LeakyReLU(0.2),     # 14 -> 7
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


print("Critic ready (no sigmoid — it outputs a real-valued score)")

<!-- cadence:task ex4-gradient-penalty -->
### Exercise 4 — The gradient penalty

Implement `gradient_penalty(C, real, fake)`: sample a per-example $\epsilon$,
form the interpolates $\hat x = \epsilon x + (1-\epsilon)\tilde x$, take the
gradient of $C(\hat x)$ w.r.t. $\hat x$ with
`torch.autograd.grad(..., create_graph=True)`, and return
$\lambda\,\overline{(\lVert\nabla\rVert_2 - 1)^2}$ with $\lambda=10$.

The cell sanity-checks your penalty on one **seeded** batch (so everyone gets the
same value) — return it as `gp_value` (a positive number). The next cell uses
your function to train the fix.

In [ ]:
def gradient_penalty(C, real, fake, lam=10.0):
    n = real.shape[0]
    eps = torch.rand(n, *([1] * (real.dim() - 1)), device=device)   # one eps per example
    # interpolate each real with its paired fake (the x_hat in the formula above):
    x_hat = ...
    x_hat.requires_grad_(True)
    score = C(x_hat)
    grad = torch.autograd.grad(
        outputs=score, inputs=x_hat, grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True)[0].reshape(n, -1)
    # penalty: lam * mean over the batch of (||grad||_2 - 1)^2  (given):
    gp = lam * ((grad.norm(2, dim=1) - 1) ** 2).mean()
    return gp

set_seed(SEED)                                # so gp_value is reproducible
_real = X[:64].to(device)
_fake = Generator().to(device)(sample_noise(64))
gp_value = round(float(gradient_penalty(Critic().to(device), _real, _fake)), 3)
print("gradient penalty on a seeded batch:", gp_value)

check("ex4-gradient-penalty", gp_value)

### The fix in action

Now we train a WGAN-GP using **your** `gradient_penalty`, then re-measure mode
coverage — it should recover the modes the collapsed GAN abandoned.

In [ ]:
set_seed(SEED)
Gw = Generator().to(device)
Cw = Critic().to(device)
ogw = torch.optim.Adam(Gw.parameters(), lr=1e-4, betas=(0.0, 0.9))
ocw = torch.optim.Adam(Cw.parameters(), lr=1e-4, betas=(0.0, 0.9))
N_CRITIC = 3
for _ in range(EPOCHS):
    for (real,) in img_loader:
        real = real.to(device); n = real.shape[0]
        for _c in range(N_CRITIC):
            fake = Gw(sample_noise(n)).detach()
            c_loss = Cw(fake).mean() - Cw(real).mean() + gradient_penalty(Cw, real, fake)
            ocw.zero_grad(); c_loss.backward(); ocw.step()
        gen = Gw(sample_noise(n))
        ogw.zero_grad(); (-Cw(gen).mean()).backward(); ogw.step()

Gw.eval()
with torch.no_grad():
    assigned = nearest_mode(Gw(sample_noise(512)), X[:1024].to(device), mode_labels[:1024])
    modes_covered_wgan = int(len(np.unique(assigned)))
print(f"WGAN-GP recovered {modes_covered_wgan}/{N_MODES} modes")

In [ ]:
# Before vs after: collapsed GAN (top) piles onto a mode or two; WGAN-GP (bottom)
# spreads back across the categories.
G_collapse.eval(); Gw.eval()
with torch.no_grad():
    coll = G_collapse(sample_noise(8)).cpu()
    fixed = Gw(sample_noise(8)).cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for row, (batch, name) in enumerate([(coll, "collapsed"), (fixed, "WGAN-GP")]):
    for j in range(8):
        axes[row, j].imshow(batch[j, 0], cmap="gray")
        axes[row, j].set_xticks([]); axes[row, j].set_yticks([])
    axes[row, 0].set_ylabel(name)
fig.suptitle("Mode collapse (top) vs the WGAN-GP fix (bottom)")
plt.show()

<!-- cadence:task ex5-wgan-collapse -->
### Exercise 5 — Why did it collapse, and why does WGAN-GP fix it? (reflection)

You watched the greedy-generator GAN collapse onto one or two modes, then WGAN-GP
spread back across the categories. Write 2–3 sentences and submit:

- Why does hammering the generator against a starved discriminator (many G steps
  per D step) drive it to pile all its mass on a single mode?
- The gradient penalty keeps the critic **1-Lipschitz**. Why do those smooth,
  bounded gradients give the generator a reason to cover *all* the modes again?

In [ ]:
# Your reflection (2-3 sentences), then run the cell to submit:
#   * why does over-training the generator against a weak D cause mode collapse?
#   * why do the critic's smooth, 1-Lipschitz gradients restore diversity?

check("ex5-wgan-collapse", "...your 2-3 sentences here...")

## Part D · A point-cloud jet GAN

Now the physics. Jets are **point clouds**, so we build a permutation-respecting
GAN: a **DeepSets generator** (`DeepSetsDecoder`: noise → 30 particles) and a
**DeepSets critic** (`DeepSetsEncoder` → score). We train it WGAN-GP-style and
overlay the **jet-mass spine** against real gluon jets. GANs are sharp in the
bulk but, like the VAE, tend to **drop the high-mass tail**.

In [ ]:
gluon = load_jetnet("g", num_particles=30, max_jets=5000)   # physical jets
JET_SHAPE = gluon.shape[1:]                 # (30, 3)
# Standardize (log-pt + z-score) so the GAN targets the right per-feature scales
# and the pt concentration; samples are inverse-transformed back to physical
# jets before the mass plot / displays.
jet_std = JetStandardizer().fit(gluon)
jet_loader = to_loader(jet_std.transform(gluon), batch_size=BATCH)
print("gluon jets:", gluon.shape, "| standardized for training")


class JetGenerator(nn.Module):
    """noise -> latent -> DeepSets decoder -> (B, 30, 3) particle cloud."""
    def __init__(self, noise=NOISE_DIM, n_particles=30):
        super().__init__()
        self.lift = nn.Sequential(nn.Linear(noise, 64), nn.ReLU(), nn.Linear(64, 32))
        self.dec = DeepSetsDecoder(latent=32, n_particles=n_particles)

    def forward(self, z):
        return self.dec(self.lift(z))


class JetCritic(nn.Module):
    """DeepSets encoder -> scalar score (1-Lipschitz target)."""
    def __init__(self):
        super().__init__()
        self.enc = DeepSetsEncoder(in_features=3, hidden=128, latent=64,
                                   mask_padding=False)   # standardized jets
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return self.head(self.enc(x)).squeeze(-1)


def jet_gradient_penalty(C, real, fake, lam=10.0):
    n = real.shape[0]
    eps = torch.rand(n, 1, 1, device=device)
    x_hat = (eps * real + (1 - eps) * fake).requires_grad_(True)
    score = C(x_hat)
    grad = torch.autograd.grad(
        outputs=score, inputs=x_hat,
        grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True)[0]
    grad = grad.reshape(n, -1)
    return lam * ((grad.norm(2, dim=1) - 1) ** 2).mean()


# See the real jets as event displays before generating any.
plot_jets(gluon[:3], titles=[f"real gluon {i}" for i in range(3)]); plt.show()
print("JetGenerator params:", sum(p.numel() for p in JetGenerator().parameters()))

<!-- cadence:task ex6-jet-plot -->
### Exercise 6 — Fix the jet GAN, then submit your jet-mass plot

Fill the two **WGAN-GP** losses in the loop, then run the cell. At the default
settings the jet-mass distribution **collapses to a spike near zero** — the
generator learns the total pt but smears it evenly across all 30 particles
instead of concentrating it on a leading particle, so every jet is nearly
massless.

Your job: **make the jet mass look believable.** The two knobs at the top of the
cell — `JET_EPOCHS` and `N_CRITIC` — set how long the generator trains and how
often the critic updates. The generator needs more of both to learn the pt
concentration. Bump them, re-run, and watch the bulk fill in (the printed
`mass_w1` should drop well below the collapsed ~0.05).

When your GAN matches the bulk — it will still under-shoot the sharp high-mass
tail, which is the physics point — the cell **submits your jet-mass plot** with
`submit_image` for the teacher to see.

> ⚠️ Training is slow on CPU, so each bump costs a few minutes. Use a GPU if you
> can, and don't over-do the epochs.

In [ ]:
# --- knobs: these defaults COLLAPSE the GAN. Increase them until the mass matches. ---
JET_EPOCHS = 20     # too few — the generator collapses to near-zero-mass jets
N_CRITIC = 3        # critic steps per generator step; more helps too

set_seed(SEED)
Gj = JetGenerator().to(device)
Cj = JetCritic().to(device)
ogj = torch.optim.Adam(Gj.parameters(), lr=1e-4, betas=(0.0, 0.9))
ocj = torch.optim.Adam(Cj.parameters(), lr=1e-4, betas=(0.0, 0.9))
for _ in range(JET_EPOCHS):
    for (real,) in jet_loader:
        real = real.to(device)
        n = real.shape[0]
        for _c in range(N_CRITIC):
            fake = Gj(sample_noise(n)).detach()
            c_loss = ...    # Cj(fake).mean() - Cj(real).mean() + jet_gradient_penalty(Cj, real, fake)
            ocj.zero_grad(); c_loss.backward(); ocj.step()
        gen = Gj(sample_noise(n))
        g_loss = ...        # -Cj(gen).mean()
        ogj.zero_grad(); g_loss.backward(); ogj.step()

Gj.eval()
with torch.no_grad():
    gen_jets = jet_std.inverse_transform(Gj(sample_noise(512)).cpu().numpy())  # -> physical jets
ax = plot_jet_mass(gluon[:512], gen_jets, labels=("real gluon", "jet GAN")); fig = ax.get_figure(); plt.show()
mass_w1 = round(jet_mass_w1(gluon[:512], gen_jets), 4)
print(f"jet-mass W1: {mass_w1}   (a collapsed GAN sits near 0.05 — aim well below)")

# once the bulk matches, submit your jet-mass plot for the teacher to see:
submit_image("ex6-jet-plot", fig)

mark_done("ex6-jet-plot")

In [ ]:
# generated jets as event displays, and overlaid on real gluon jets:
plot_jets(gen_jets[:3], titles=[f"GAN jet {i}" for i in range(3)]); plt.show()
plot_jet_overlay(gluon[:3], gen_jets[:3], n=3, labels=("real gluon", "GAN jet")); plt.show()

In [ ]:
import numpy as np
def stats(jets, name):
    j = np.asarray(jets); m = j[..., 2] > 0
    print(f"{name:10} | eta std {j[...,0][m].std():.3f}  phi std {j[...,1][m].std():.3f}  "
          f"lead-pt {j[...,2].max(1).mean():.3f}  mult {m.sum(1).mean():.1f}")
stats(gluon[:512], "real")
stats(gen_jets,    "generated")    # or recon_g in notebook 01

<!-- cadence:task ex7-jet-tail -->
### Exercise 7 — Where does the jet GAN fall short? (reflection)

Look at the jet-mass spine and the feature stats you just printed (the generated
jets come out narrower — smaller η/φ spread, lower lead-pt). Write 2–3 sentences
and submit:

- Does the GAN match the real gluon jets better in the **bulk** or the **tail**
  of the mass distribution?
- How does making jets too narrow bias the jet-mass, and why is the sharp,
  physical tail the hardest part for a GAN (and the VAE in notebook 01) to get?

In [ ]:
# Your reflection (2-3 sentences), then run the cell to submit:
#   * bulk or tail — where does the GAN match the real jets worst?
#   * how do too-narrow jets bias the mass, and why is the tail so hard?

check("ex7-jet-tail", "...your 2-3 sentences here...")

## Recap

- A GAN is an **adversarial game**: generator vs discriminator/critic.
- **Mode collapse** is the signature failure — the generator covers only a few
  data modes (we *measured* it by snapping samples to nearest data modes).
- **WGAN-GP** — a 1-Lipschitz critic enforced by a **gradient penalty** — gives
  smooth gradients and restores diversity.
- On jets, a **DeepSets generator + critic** matches the bulk of the jet-mass
  spine but **drops the high-mass tail**, the same hard region the VAE missed in
  01 — a story we'll revisit with diffusion (03, 05).